<a href="https://colab.research.google.com/github/likeshd/case_studies_and_projects/blob/main/rank_trend_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:

# Load the uploaded Excel file to inspect the structure of the data
file_path = '/content/rank_trend.xlsx'
rank_data = pd.read_excel(file_path)

# Display the first few rows of the data to understand its structure
rank_data.head()

,keyword,product id,rank,rank_date
0,ghanaian bowl,185289306,45,2023-11-03
1,ghanaian bowl,999587744,44,2023-11-03
2,ghanaian bowl,810487529,43,2023-11-03
3,ghanaian bowl,442935243,42,2023-11-03
4,ghanaian bowl,1880783853,41,2023-11-03


In [ ]:
rank_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4547 entries, 0 to 4546
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   keyword     4547 non-null   object        
 1   product id  4547 non-null   int64         
 2   rank        4547 non-null   int64         
 3   rank_date   4547 non-null   datetime64[ns]
dtypes: datetime64[ns](1), int64(2), object(1)
memory usage: 142.2+ KB


In [ ]:
rank_data["keyword"].value_counts()

,count
keyword,
ghanaian bowl,559
handmade bowl,544
reusable bath sponge,541
bowl for scrub,540
hand crafted bowl,518
calabash cup,491
shower body scrubber,464
african bowl,451
facial bowl,439


In [ ]:
rank_data.describe()

,product id,rank,rank_date
count,4.547000e+03,4547.000000,4547
mean,7.489738e+08,23.792391,2023-09-25 19:09:16.586760704
min,1.029434e+07,1.000000,2023-08-18 00:00:00
25%,3.594688e+08,12.000000,2023-09-05 00:00:00
50%,6.785312e+08,24.000000,2023-09-26 00:00:00
75%,9.748549e+08,35.000000,2023-10-17 00:00:00
max,1.997448e+09,50.000000,2023-11-03 00:00:00
std,4.972975e+08,13.706030,NaN


In [ ]:
# Check for missing values
missing_summary = rank_data.isnull().sum()
missing_summary


,0
keyword,0
product id,0
rank,0
rank_date,0


In [ ]:
# Check for duplicates
duplicates = rank_data.duplicated().sum()
duplicates

0

In [ ]:
# Check for outliers in ranks (IQR method)
q1 = rank_data['rank'].quantile(0.25)
q3 = rank_data['rank'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = rank_data[(rank_data['rank'] < lower_bound) | (rank_data['rank'] > upper_bound)]
print(f"Number of outliers in rank: {len(outliers)}")


Number of outliers in rank: 0


In [ ]:
# Number of unique keywords and product IDs
print(f"Number of unique keywords: {rank_data['keyword'].nunique()}")
print(f"Number of unique product IDs: {rank_data['product id'].nunique()}")

Number of unique keywords: 9
Number of unique product IDs: 1479


In [ ]:
# Step 1: Trend Analysis
# Recalculate trends for cleaned data
def calculate_trend(group):
    if len(group) < 2:
        return np.nan  # Trend can't be calculated with a single data point
    X = (group['rank_date'] - group['rank_date'].min()).dt.days.values.reshape(-1, 1)
    y = group['rank'].values
    model = LinearRegression()
    model.fit(X, y)
    return model.coef_[0]  # Return the slope

In [ ]:
# Calculate trends for each product id
trends_cleaned = rank_data.groupby('product id').apply(calculate_trend).reset_index(name='trend')
trends_cleaned

<ipython-input-43-ce11dd787fb2>:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  trends_cleaned = rank_data.groupby('product id').apply(calculate_trend).reset_index(name='trend')


,product id,trend
0,10294336,NaN
1,10535909,0.285714
2,12443821,NaN
3,14118807,0.233871
4,14118808,0.131677
...,...,...
1474,1985208473,NaN
1475,1986456303,NaN
1476,1987486982,NaN
1477,1988819054,0.080000


In [ ]:
# Classify product trends as Positive or Negative
trends_cleaned['trend_type'] = np.where(trends_cleaned['trend'] > 0, 'Negative', 'Positive')

In [ ]:
trends_cleaned

,product id,trend,trend_type
0,10294336,NaN,Positive
1,10535909,0.285714,Negative
2,12443821,NaN,Positive
3,14118807,0.233871,Negative
4,14118808,0.131677,Negative
...,...,...,...
1474,1985208473,NaN,Positive
1475,1986456303,NaN,Positive
1476,1987486982,NaN,Positive
1477,1988819054,0.080000,Negative


In [ ]:

# Extract product ids based on trend type
positive_trend_products = trends_cleaned[trends_cleaned['trend_type'] == 'Positive']['product id'].tolist()
negative_trend_products = trends_cleaned[trends_cleaned['trend_type'] == 'Negative']['product id'].tolist()


In [ ]:
# positive_trend_products, negative_trend_products

In [ ]:
# Step 2: Prediction for common product ids per keyword
predictions_cleaned = []
for keyword, group in rank_data.groupby('keyword'):
    common_product_ids = group['product id'].value_counts()[lambda x: x > 1].index
    for product_id in common_product_ids:
        product_data = group[group['product id'] == product_id]
        if len(product_data) < 2:
            continue  # Skip if not enough data points for prediction

        # Prepare data for prediction
        X = (product_data['rank_date'] - product_data['rank_date'].min()).dt.days.values.reshape(-1, 1)
        y = product_data['rank'].values
        model = LinearRegression()
        model.fit(X, y)

        # Predict the next rank
        next_day = (product_data['rank_date'].max() - product_data['rank_date'].min()).days + 1
        predicted_rank = model.predict([[next_day]])[0]

        # Append prediction to results
        predictions_cleaned.append({
            'keyword': keyword,
            'product id': product_id,
            'predicted_rank': predicted_rank
        })

predictions_cleaned_df = pd.DataFrame(predictions_cleaned)

In [ ]:
predictions_cleaned_df

,keyword,product id,predicted_rank
0,african bowl,1351725243,24.151893
1,african bowl,1613336318,20.796685
2,african bowl,264243105,7.722043
3,african bowl,697862250,20.668074
4,african bowl,117105357,40.458572
...,...,...,...
849,shower body scrubber,658598712,50.604651
850,shower body scrubber,933100234,9.206897
851,shower body scrubber,906814149,40.100000
852,shower body scrubber,337222839,34.180000
